<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/insurance_claim_agent_mockingDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27
Found existing installation: langchain-core 0.3.84
Uninstalling langchain-core-0.3.84:
  Successfully uninstalled langchain-core-0.3.84
Found existing installation: langchain-community 0.3.24
Uninstalling langchain-community-0.3.24:
  Successfully uninstalled langchain-community-0.3.24
  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_community-0.3.24-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain-0.3.27-py3-none-any.whl (1.0 MB)
Using cached langchain_community-0.3.24-py3-none-any.whl (2.5 MB)
Using cached langchain_core-0.3.84-py3-none-any.whl (459 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt

In [2]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


# In this example: An Insurance Claims Assistant we using mock api tool for check claim status, Fraud Detection Tool, Escalation tool



In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import os
from google.colab import userdata

# Load environment variables
load_dotenv()

# Retrieve the OpenAI API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Create model
llm = ChatOpenAI(model="gpt-4o-mini")

In [5]:
from langchain.tools import tool

# Mock API Tool: Check Claim Status
@tool
def check_claim_status(claim_id: str) -> str:
    """Return status of a claim with given claim_id."""
    fake_db = {
        "C101": "Approved - Payment in process",
        "C202": "Under review - Waiting documents",
        "C303": "Flagged for manual investigation"
    }
    return fake_db.get(claim_id, "Claim ID not found")

# Fraud Detection Tool
@tool
def fraud_risk_check(description: str) -> str:
    """Returns risk evaluation for fraud based on description."""
    if "lost phone but still using it" in description.lower():
        return "High Risk"
    return "Low Risk"

# Escalation Tool
@tool
def escalate_to_human(issue: str) -> str:
    """Escalates conversation to a human agent."""
    return f"Escalation created. A human agent will review: {issue}"


In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain.prompts import ChatPromptTemplate


tools = [check_claim_status, fraud_risk_check, escalate_to_human]

system_prompt = ChatPromptTemplate.from_template("""
You are an Insurance Claims Assistant.

Your job:
1) Understand user intent.
2) If they ask about a claim status → call `check_claim_status`
3) If user mentions incident details → call `fraud_risk_check`
4) If unsure, conflicting, or user upset → call `escalate_to_human`

Always respond in a friendly, formal tone.
user request: {input},
"placeholder", "{agent_scratchpad}"
"""
)



In [7]:
from openai import max_retries


agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=system_prompt
)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)


In [8]:
# Test the agent with a sample query
query = "What's the current status of claim C202?"
result = executor.invoke({"input": query})
print("Agent Output:", result)



> Entering new AgentExecutor chain...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
#test fraud_risk_check tools: "I lost my phone but I'm still using it. Is that normal?"
#test escalate_to_human: "Your bot is useless. I want a human NOW!"


In [ ]:
# Test the agent with a query for fraud_risk_check
query_fraud = "I lost my phone but I'm still using it. Is that normal?"
result_fraud = executor.invoke({"input": query_fraud})
print("Agent Output (Fraud Risk Check):", result_fraud)



> Entering new AgentExecutor chain...

Invoking: `escalate_to_human` with `{'issue': 'User is confused about losing their phone but still using it.'}`


Escalation created. A human agent will review: User is confused about losing their phone but still using it.It seems that you are experiencing some confusion regarding your phone situation. While it's a common scenario to think you've lost a phone when it’s just not in your immediate view, being able to use it suggests you may still have it. However, to ensure you get the right assistance and clarification, I will escalate this matter to a human agent who can help you further. They will review your situation and provide guidance. Thank you for your understanding!

> Finished chain.
Agent Output (Fraud Risk Check): {'input': "I lost my phone but I'm still using it. Is that normal?", 'output': "It seems that you are experiencing some confusion regarding your phone situation. While it's a common scenario to think you've lost a phone whe

In [ ]:
# Test the agent with a query for escalate_to_human
query_escalate = "Your bot is useless. I want a human NOW!"
result_escalate = executor.invoke({"input": query_escalate})
print("Agent Output (Escalate to Human):", result_escalate)



> Entering new AgentExecutor chain...

Invoking: `escalate_to_human` with `{'issue': 'User expressed frustration and requested to speak with a human agent.'}`


Escalation created. A human agent will review: User expressed frustration and requested to speak with a human agent.
Invoking: `escalate_to_human` with `{'issue': 'User expressed frustration and requested to speak with a human agent.'}`


Escalation created. A human agent will review: User expressed frustration and requested to speak with a human agent.
Invoking: `escalate_to_human` with `{'issue': 'User expressed frustration and requested to speak with a human agent.'}`


Escalation created. A human agent will review: User expressed frustration and requested to speak with a human agent.
Invoking: `escalate_to_human` with `{'issue': 'User expressed frustration and requested to speak with a human agent.'}`


Escalation created. A human agent will review: User expressed frustration and requested to speak with a human agent.
Inv